In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Consistent styling
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Architecture color palette (consistent across all plots)
ARCH_COLORS = {
    'TrendWaveletAELG': '#2196F3',       # Blue
    'TrendWaveletAE': '#4CAF50',          # Green
    'GenericAE': '#FF9800',               # Orange
    'TrendVAE_HaarWaveletV3': '#9C27B0',  # Purple
    'TrendVAE_WaveletV3VAE': '#F44336',   # Red
    'TrendVAE2_WaveletV3VAE2': '#795548', # Brown
}

ARCH_SHORT = {
    'TrendWaveletAELG': 'TWALG',
    'TrendWaveletAE': 'TWA',
    'GenericAE': 'GAE',
    'TrendVAE_HaarWaveletV3': 'TVH',
    'TrendVAE_WaveletV3VAE': 'TVAE',
    'TrendVAE2_WaveletV3VAE2': 'TV2',
}

# Load data
CSV_PATH = '/home/realdanielbyrne/GitHub/N-BEATS-Lightning/experiments/results/m4/resnet_skip_study_v2_results.csv'
df = pd.read_csv(CSV_PATH)

# Parse n_stacks as int
df['n_stacks'] = df['n_stacks'].astype(int)
df['skip_distance_cfg'] = df['skip_distance_cfg'].astype(int)
df['search_round'] = df['search_round'].astype(int)

# Add derived columns
df['has_skip'] = df['skip_distance_cfg'] > 0
df['arch_short'] = df['architecture'].map(ARCH_SHORT)

# Split by round
r1 = df[df['search_round'] == 1].copy()
r2 = df[df['search_round'] == 2].copy()
r3 = df[df['search_round'] == 3].copy()

print(f"Total runs: {len(df)}")
print(f"Round 1: {r1['config_name'].nunique()} configs, {len(r1)} runs (15 epochs)")
print(f"Round 2: {r2['config_name'].nunique()} configs, {len(r2)} runs (30 epochs)")
print(f"Round 3: {r3['config_name'].nunique()} configs, {len(r3)} runs (100 epochs, early stop)")
print(f"\nArchitectures: {df['architecture'].nunique()}")
print(f"Diverged runs: {df['diverged'].sum()}")

# ResNet Skip Connection Study v2 -- Comprehensive Analysis

**Dataset:** M4-Yearly (H=6, L=30)  
**Date:** 2026-03-07  
**Method:** 3-round successive halving (36 configs: 15/30/100 epochs)  
**Results:** `experiments/results/m4/resnet_skip_study_v2_results.csv`  
**Config:** `experiments/configs/resnet_skip_study_v2.yaml`

## Study Motivation

The v1 skip connection study (24 configs) established that skip connections rescue **GenericAELG** at depth >= 20 stacks (SMAPE 36 -> 13.8) but do not help TrendAELG+WaveletV3AELG or Generic, which are inherently depth-stable. This v2 follow-up fills gaps identified in v1:

| Section | Architecture | Configs | Question |
|---------|-------------|---------|----------|
| A | GenericAE (homogeneous) | 6 | Does GenericAE degrade at depth like GenericAELG? |
| B | TrendVAE + WaveletV3VAE | 6 | Does double-VAE interact with skip? |
| C | TrendVAE2 + WaveletV3VAE2 | 6 | Does double-VAE2 (known unstable) benefit from skip? |
| D | TrendWaveletAE (homogeneous) | 6 | Combined block depth scaling + short skip distances |
| E | TrendWaveletAELG (homogeneous) | 6 | LG variant depth scaling -- gate effect vs. non-LG |
| F | TrendVAE + HaarWaveletV3 | 6 | Corrected single-VAE design (VAE + deterministic wavelet) |

## 1. All Results Overview

In [ ]:
# Summary table: all configs, aggregated across all rounds
summary_all = df.groupby(['config_name', 'architecture', 'n_stacks', 'skip_distance_cfg']).agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    mean_owa=('owa', 'mean'),
    n_runs=('smape', 'count'),
    max_round=('search_round', 'max'),
    mean_params=('n_params', 'first'),
).sort_values('mean_smape').reset_index()

summary_all['cv_pct'] = (summary_all['std_smape'] / summary_all['mean_smape'] * 100).round(1)
summary_all['skip'] = summary_all['skip_distance_cfg'].apply(lambda x: f'd={x}' if x > 0 else '--')

display_cols = ['config_name', 'architecture', 'n_stacks', 'skip', 'mean_smape', 'std_smape', 'cv_pct', 'mean_owa', 'n_runs', 'max_round', 'mean_params']
print("All 36 Configs Ranked by Mean SMAPE (all rounds pooled):")
print("=" * 120)
print(summary_all[display_cols].to_string(index=False, float_format='%.3f'))

## 2. R3 Final Rankings (Definitive Results)

In [ ]:
# R3 final rankings bar chart
r3_summary = r3.groupby(['config_name', 'architecture']).agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    mean_owa=('owa', 'mean'),
).sort_values('mean_smape').reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
colors = [ARCH_COLORS[a] for a in r3_summary['architecture']]
bars = ax.barh(range(len(r3_summary)), r3_summary['mean_smape'], 
               xerr=r3_summary['std_smape'], capsize=3, color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(r3_summary)))
ax.set_yticklabels(r3_summary['config_name'], fontsize=10)
ax.set_xlabel('SMAPE')
ax.set_title('Round 3 Final Rankings (100 epochs, 5 runs each)')
ax.invert_yaxis()

# Add value labels
for i, (s, std) in enumerate(zip(r3_summary['mean_smape'], r3_summary['std_smape'])):
    ax.text(s + std + 0.02, i, f'{s:.3f}', va='center', fontsize=9)

# Legend
patches = [mpatches.Patch(color=c, label=ARCH_SHORT[a]) for a, c in ARCH_COLORS.items() 
           if a in r3_summary['architecture'].values]
ax.legend(handles=patches, loc='lower right', framealpha=0.9)
plt.tight_layout()
plt.show()

## 3. Successive Halving Progression

Which configs survived each round? The halving should eliminate poorly performing architectures early.

In [ ]:
# Successive halving Sankey-style visualization
r1_configs = set(r1['config_name'].unique())
r2_configs = set(r2['config_name'].unique())
r3_configs = set(r3['config_name'].unique())

# Get mean SMAPE per config per round for ordering
def round_summary(rdf):
    return rdf.groupby(['config_name', 'architecture'])['smape'].mean().reset_index().sort_values('smape')

r1_sum = round_summary(r1)
r2_sum = round_summary(r2)
r3_sum = round_summary(r3)

fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharey=False)

for ax, rsum, title, n_total in zip(axes, [r1_sum, r2_sum, r3_sum],
    ['Round 1 (15 ep, 36 configs)', 'Round 2 (30 ep, 18 configs)', 'Round 3 (100 ep, 9 configs)'],
    [36, 18, 9]):
    
    colors = [ARCH_COLORS.get(a, '#999') for a in rsum['architecture']]
    survived_r2 = [c in r2_configs for c in rsum['config_name']]
    survived_r3 = [c in r3_configs for c in rsum['config_name']]
    
    bars = ax.barh(range(len(rsum)), rsum['smape'], color=colors, edgecolor='white', linewidth=0.5)
    
    # Mark eliminated configs with hatching
    for i, (bar, cfg) in enumerate(zip(bars, rsum['config_name'])):
        if rsum is r1_sum and cfg not in r2_configs:
            bar.set_hatch('///')
            bar.set_alpha(0.4)
        elif rsum is r2_sum and cfg not in r3_configs:
            bar.set_hatch('///')
            bar.set_alpha(0.4)
    
    ax.set_yticks(range(len(rsum)))
    ax.set_yticklabels(rsum['config_name'], fontsize=7)
    ax.set_xlabel('Mean SMAPE')
    ax.set_title(title, fontsize=11)
    ax.invert_yaxis()

# Legend
patches = [mpatches.Patch(color=c, label=f'{ARCH_SHORT[a]} ({a})') for a, c in ARCH_COLORS.items()]
patches.append(mpatches.Patch(facecolor='gray', hatch='///', alpha=0.4, label='Eliminated'))
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Successive Halving Progression', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Print elimination summary
print("\nEliminated after R1 (18):")
for c in sorted(r1_configs - r2_configs):
    arch = r1[r1['config_name']==c]['architecture'].iloc[0]
    smape = r1[r1['config_name']==c]['smape'].mean()
    print(f"  {c:25s} ({ARCH_SHORT[arch]})  SMAPE={smape:.2f}")

print(f"\nEliminated after R2 (9):")
for c in sorted(r2_configs - r3_configs):
    arch = r2[r2['config_name']==c]['architecture'].iloc[0]
    smape = r2[r2['config_name']==c]['smape'].mean()
    print(f"  {c:25s} ({ARCH_SHORT[arch]})  SMAPE={smape:.2f}")

## 4. Per-Architecture Box Plots

Comparing SMAPE distributions across all configs, grouped by architecture family. Uses R1 data (all 36 configs) for the broadest view.

In [ ]:
# Box plots by architecture (R1 only -- all 36 configs present)
arch_order = r1.groupby('architecture')['smape'].mean().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(14, 7))
bp_data = [r1[r1['architecture'] == a]['smape'].values for a in arch_order]
bp = ax.boxplot(bp_data, labels=[ARCH_SHORT[a] for a in arch_order], 
                patch_artist=True, widths=0.6, showfliers=True,
                flierprops=dict(marker='o', markersize=4, alpha=0.5))

for patch, arch in zip(bp['boxes'], arch_order):
    patch.set_facecolor(ARCH_COLORS[arch])
    patch.set_alpha(0.7)

# Overlay individual points
for i, (arch, data) in enumerate(zip(arch_order, bp_data)):
    jitter = np.random.normal(0, 0.05, len(data))
    ax.scatter(np.full_like(data, i+1) + jitter, data, 
               color=ARCH_COLORS[arch], alpha=0.4, s=20, zorder=3)

ax.set_ylabel('SMAPE')
ax.set_title('SMAPE Distribution by Architecture (Round 1, all configs)')
ax.set_xlabel('Architecture')

# Add median labels
for i, data in enumerate(bp_data):
    med = np.median(data)
    ax.text(i+1, med - 0.5, f'{med:.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Skip vs No-Skip Paired Comparison

For each architecture that has both skip and no-skip variants at the same depth, compare the effect of skip connections.

In [ ]:
# Skip vs no-skip paired comparison
# Use highest-round data available for each config
def get_best_round(cfg_name):
    sub = df[df['config_name'] == cfg_name]
    max_round = sub['search_round'].max()
    return sub[sub['search_round'] == max_round]

# Define pairs: (no_skip_config, skip_config, architecture, stacks, skip_dist)
pairs = [
    # GenericAE
    ('GAE30_no_skip', 'GAE30_skip3_a01', 'GenericAE', 30, 3),
    ('GAE20_no_skip', 'GAE20_skip5_a01', 'GenericAE', 20, 5),
    ('GAE30_no_skip', 'GAE30_skip5_a01', 'GenericAE', 30, 5),
    # TrendWaveletAE
    ('TWA20_no_skip', 'TWA20_skip3_a01', 'TrendWaveletAE', 20, 3),
    ('TWA30_no_skip', 'TWA30_skip3_a01', 'TrendWaveletAE', 30, 3),
    ('TWA30_no_skip', 'TWA30_skip2_a01', 'TrendWaveletAE', 30, 2),
    # TrendWaveletAELG
    ('TWALG20_no_skip', 'TWALG20_skip3_a01', 'TrendWaveletAELG', 20, 3),
    ('TWALG30_no_skip', 'TWALG30_skip3_a01', 'TrendWaveletAELG', 30, 3),
    ('TWALG30_no_skip', 'TWALG30_skip2_a01', 'TrendWaveletAELG', 30, 2),
    # TrendVAE+WaveletV3VAE
    ('TVAE16_no_skip', 'TVAE16_skip4_a01', 'TrendVAE_WaveletV3VAE', 16, 4),
    ('TVAE24_no_skip', 'TVAE24_skip4_a01', 'TrendVAE_WaveletV3VAE', 24, 4),
    ('TVAE24_no_skip', 'TVAE24_skip3_a01', 'TrendVAE_WaveletV3VAE', 24, 3),
    # TrendVAE+HaarWaveletV3
    ('TVH20_no_skip', 'TVH20_skip5_a01', 'TrendVAE_HaarWaveletV3', 20, 5),
    ('TVH30_no_skip', 'TVH30_skip5_a01', 'TrendVAE_HaarWaveletV3', 30, 5),
    ('TVH30_no_skip', 'TVH30_skip3_a01', 'TrendVAE_HaarWaveletV3', 30, 3),
]

fig, ax = plt.subplots(figsize=(14, 8))
y_labels = []
deltas = []
colors_list = []

for i, (base_cfg, skip_cfg, arch, stacks, sdist) in enumerate(pairs):
    base_data = r1[r1['config_name'] == base_cfg]['smape']
    skip_data = r1[r1['config_name'] == skip_cfg]['smape']
    
    if len(base_data) == 0 or len(skip_data) == 0:
        continue
    
    delta = skip_data.mean() - base_data.mean()
    deltas.append(delta)
    y_labels.append(f'{ARCH_SHORT[arch]}{stacks} d={sdist}')
    colors_list.append(ARCH_COLORS[arch])

bars = ax.barh(range(len(deltas)), deltas, color=colors_list, edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(range(len(deltas)))
ax.set_yticklabels(y_labels, fontsize=10)
ax.set_xlabel('Delta SMAPE (skip - no_skip)')
ax.set_title('Effect of Skip Connections: Negative = Skip Helps')

for i, d in enumerate(deltas):
    side = 'left' if d > 0 else 'right'
    offset = 0.1 if d > 0 else -0.1
    ax.text(d + offset, i, f'{d:+.2f}', va='center', ha=side, fontsize=9)

plt.tight_layout()
plt.show()

## 6. Depth Scaling Analysis

How does SMAPE change as we increase stack depth from 10 to 30 for each architecture?

In [ ]:
# Depth scaling: SMAPE vs n_stacks for no-skip baselines
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Deterministic architectures (R1 baselines only)
ax = axes[0]
for arch in ['TrendWaveletAE', 'TrendWaveletAELG', 'GenericAE']:
    baseline = r1[(r1['architecture'] == arch) & (~r1['has_skip'])]
    if len(baseline) == 0:
        continue
    agg = baseline.groupby('n_stacks').agg(mean=('smape','mean'), std=('smape','std')).reset_index()
    ax.errorbar(agg['n_stacks'], agg['mean'], yerr=agg['std'], 
                marker='o', label=ARCH_SHORT[arch], color=ARCH_COLORS[arch],
                linewidth=2, capsize=4, markersize=8)

ax.set_xlabel('Number of Stacks')
ax.set_ylabel('SMAPE')
ax.set_title('Depth Scaling: Deterministic Architectures')
ax.legend()
ax.set_xticks([10, 20, 30])
ax.grid(alpha=0.3)

# Panel 2: VAE architectures
ax = axes[1]
for arch in ['TrendVAE_WaveletV3VAE', 'TrendVAE2_WaveletV3VAE2', 'TrendVAE_HaarWaveletV3']:
    baseline = r1[(r1['architecture'] == arch) & (~r1['has_skip'])]
    if len(baseline) == 0:
        continue
    agg = baseline.groupby('n_stacks').agg(mean=('smape','mean'), std=('smape','std')).reset_index()
    ax.errorbar(agg['n_stacks'], agg['mean'], yerr=agg['std'], 
                marker='s', label=ARCH_SHORT[arch], color=ARCH_COLORS[arch],
                linewidth=2, capsize=4, markersize=8)

ax.set_xlabel('Number of Stacks')
ax.set_ylabel('SMAPE')
ax.set_title('Depth Scaling: VAE Architectures')
ax.legend()
ax.grid(alpha=0.3)

# Panel 3: All architectures at 10 stacks (or lowest) -- normalized comparison
ax = axes[2]
arch_best = {}
for arch in ARCH_COLORS.keys():
    baseline = r1[(r1['architecture'] == arch) & (~r1['has_skip'])]
    if len(baseline) > 0:
        agg = baseline.groupby('n_stacks')['smape'].mean()
        arch_best[ARCH_SHORT[arch]] = agg.min()

sorted_archs = sorted(arch_best.items(), key=lambda x: x[1])
names = [a[0] for a in sorted_archs]
vals = [a[1] for a in sorted_archs]
full_names = {v: k for k, v in ARCH_SHORT.items()}
colors = [ARCH_COLORS[full_names[n]] for n in names]

ax.barh(range(len(names)), vals, color=colors, edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Best Baseline SMAPE (any depth)')
ax.set_title('Best No-Skip Performance by Architecture')
ax.invert_yaxis()
for i, v in enumerate(vals):
    ax.text(v + 0.2, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Double-VAE vs Single-VAE vs Deterministic

A critical finding: pairing two VAE-backbone blocks compounds stochastic noise and destroys performance.

In [ ]:
# VAE design comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Bar chart of matched-depth comparisons
ax = axes[0]
comparisons = {
    'Double-VAE\n(TVAE, 8 stacks)': ('TVAE8_no_skip', r1),
    'Double-VAE2\n(TV2, 8 stacks)': ('TV2_8_no_skip', r1),
    'Single-VAE\n(TVH, 10 stacks)': ('TVH10_no_skip', r1),
    'TrendWaveletAE\n(TWA, 10 stacks)': ('TWA10_no_skip', r1),
    'TrendWaveletAELG\n(TWALG, 10 stacks)': ('TWALG10_no_skip', r1),
}

names = list(comparisons.keys())
means = []
stds = []
bar_colors = []
arch_map = {'TVAE': 'TrendVAE_WaveletV3VAE', 'TV2': 'TrendVAE2_WaveletV3VAE2',
            'TVH': 'TrendVAE_HaarWaveletV3', 'TWA': 'TrendWaveletAE', 'TWALG': 'TrendWaveletAELG'}

for label, (cfg, rdf) in comparisons.items():
    sub = rdf[rdf['config_name'] == cfg]
    means.append(sub['smape'].mean())
    stds.append(sub['smape'].std())
    # Get arch from first 3-4 chars
    for short, full in arch_map.items():
        if cfg.startswith(short):
            bar_colors.append(ARCH_COLORS[full])
            break

bars = ax.bar(range(len(names)), means, yerr=stds, capsize=5, color=bar_colors, 
              edgecolor='white', linewidth=0.5, alpha=0.85)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('SMAPE')
ax.set_title('VAE Design Comparison (Smallest Stacks)')
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.5, f'{m:.1f}', ha='center', fontsize=10, fontweight='bold')

# Panel 2: Parameter efficiency
ax = axes[1]
param_data = []
for label, (cfg, rdf) in comparisons.items():
    sub = rdf[rdf['config_name'] == cfg]
    param_data.append({
        'label': label.replace('\n', ' '),
        'smape': sub['smape'].mean(),
        'params': sub['n_params'].iloc[0] / 1e6,
        'cfg': cfg,
    })

pdf = pd.DataFrame(param_data)
for _, row in pdf.iterrows():
    for short, full in arch_map.items():
        if row['cfg'].startswith(short):
            color = ARCH_COLORS[full]
            break
    ax.scatter(row['params'], row['smape'], s=150, color=color, edgecolors='black', linewidth=0.5, zorder=3)
    ax.annotate(row['label'], (row['params'], row['smape']), fontsize=8,
                xytext=(8, 4), textcoords='offset points')

ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('SMAPE')
ax.set_title('Parameter Efficiency: SMAPE vs Model Size')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Double-VAE (SMAPE ~29-37) is 2-3x worse than single-VAE (SMAPE ~15)")
print("Single-VAE + deterministic wavelet still underperforms deterministic-only (SMAPE ~14)")
print("TrendWaveletAE achieves best SMAPE with fewest parameters")

## 8. Parameter Efficiency Analysis

SMAPE vs parameter count for all configs. Are bigger models better?

In [ ]:
# Parameter efficiency: SMAPE vs params for all R1 configs
r1_agg = r1.groupby(['config_name', 'architecture', 'n_stacks', 'has_skip']).agg(
    mean_smape=('smape', 'mean'),
    n_params=('n_params', 'first'),
).reset_index()

fig, ax = plt.subplots(figsize=(14, 8))

for arch in ARCH_COLORS:
    sub = r1_agg[r1_agg['architecture'] == arch]
    if len(sub) == 0:
        continue
    
    skip_mask = sub['has_skip']
    # No-skip: filled circles
    no_skip = sub[~skip_mask]
    if len(no_skip) > 0:
        ax.scatter(no_skip['n_params']/1e6, no_skip['mean_smape'], 
                   color=ARCH_COLORS[arch], s=100, marker='o', edgecolors='black',
                   linewidth=0.5, label=f'{ARCH_SHORT[arch]} (no skip)', zorder=3)
    # Skip: open diamonds
    with_skip = sub[skip_mask]
    if len(with_skip) > 0:
        ax.scatter(with_skip['n_params']/1e6, with_skip['mean_smape'], 
                   color=ARCH_COLORS[arch], s=100, marker='D', edgecolors='black',
                   linewidth=0.5, label=f'{ARCH_SHORT[arch]} (skip)', zorder=3, alpha=0.7)

# Annotate extremes
for _, row in r1_agg.iterrows():
    if row['mean_smape'] < 14.5 or row['mean_smape'] > 35 or row['n_params'] > 10e6:
        ax.annotate(row['config_name'], (row['n_params']/1e6, row['mean_smape']),
                    fontsize=7, xytext=(5, 3), textcoords='offset points', alpha=0.8)

ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('Mean SMAPE')
ax.set_title('Parameter Efficiency: All R1 Configs')
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.grid(alpha=0.3)

# Draw Pareto frontier
pareto = r1_agg.sort_values('n_params')
pareto_front = []
best_so_far = float('inf')
for _, row in pareto.iterrows():
    if row['mean_smape'] < best_so_far:
        pareto_front.append(row)
        best_so_far = row['mean_smape']

if pareto_front:
    pf = pd.DataFrame(pareto_front)
    ax.step(pf['n_params']/1e6, pf['mean_smape'], where='post', 
            color='gray', linewidth=1.5, linestyle='--', alpha=0.5, label='Pareto frontier')

plt.tight_layout()
plt.show()

## 9. Stability Analysis (Coefficient of Variation)

Low CV indicates a reliable, reproducible configuration.

In [ ]:
# Stability analysis: CV% for R3 configs
r3_stab = r3.groupby(['config_name', 'architecture']).agg(
    mean_smape=('smape', 'mean'),
    std_smape=('smape', 'std'),
    mean_owa=('owa', 'mean'),
    std_owa=('owa', 'std'),
).reset_index()

r3_stab['cv_smape'] = (r3_stab['std_smape'] / r3_stab['mean_smape'] * 100)
r3_stab['cv_owa'] = (r3_stab['std_owa'] / r3_stab['mean_owa'] * 100)
r3_stab = r3_stab.sort_values('cv_smape')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CV of SMAPE
ax = axes[0]
colors = [ARCH_COLORS[a] for a in r3_stab['architecture']]
ax.barh(range(len(r3_stab)), r3_stab['cv_smape'], color=colors, edgecolor='white')
ax.set_yticks(range(len(r3_stab)))
ax.set_yticklabels(r3_stab['config_name'], fontsize=10)
ax.set_xlabel('CV% (SMAPE)')
ax.set_title('SMAPE Stability: Lower = More Reproducible')
ax.invert_yaxis()
for i, cv in enumerate(r3_stab['cv_smape']):
    ax.text(cv + 0.05, i, f'{cv:.1f}%', va='center', fontsize=9)
ax.axvline(2.0, color='red', linestyle='--', alpha=0.5, label='2% threshold')
ax.legend()

# CV of OWA
ax = axes[1]
r3_stab_owa = r3_stab.sort_values('cv_owa')
colors = [ARCH_COLORS[a] for a in r3_stab_owa['architecture']]
ax.barh(range(len(r3_stab_owa)), r3_stab_owa['cv_owa'], color=colors, edgecolor='white')
ax.set_yticks(range(len(r3_stab_owa)))
ax.set_yticklabels(r3_stab_owa['config_name'], fontsize=10)
ax.set_xlabel('CV% (OWA)')
ax.set_title('OWA Stability: Lower = More Reproducible')
ax.invert_yaxis()
for i, cv in enumerate(r3_stab_owa['cv_owa']):
    ax.text(cv + 0.05, i, f'{cv:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nMost stable (CV < 1%): TWA30_no_skip (0.5%), TWALG30_skip3 (0.5%), TWALG10_no_skip (0.7%)")
print("Least stable: TWALG20_no_skip (4.3%) -- the LG gate adds variance at 20 stacks")

## 10. Cross-Reference with v1 Results

How do v2 winners compare to the v1 skip study findings?

In [ ]:
# v1 vs v2 comparison
# v1 results from resnet_skip_study_analysis.md
v1_results = {
    'TW16_skip4_learn': {'smape': 13.521, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 16, 'skip': 'd=4,learn'},
    'TW16_skip4_a01':   {'smape': 13.557, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 16, 'skip': 'd=4,a=0.1'},
    'TW24_no_skip':     {'smape': 13.557, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 24, 'skip': 'none'},
    'TW8_no_skip':      {'smape': 13.601, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 8, 'skip': 'none'},
    'TW16_no_skip':     {'smape': 13.699, 'arch': 'TrendAELG+WaveletV3AELG', 'stacks': 16, 'skip': 'none'},
    'GAELG10_no_skip':  {'smape': 13.823, 'arch': 'GenericAELG', 'stacks': 10, 'skip': 'none'},
    'GAELG30_skip5_a01':{'smape': 13.786, 'arch': 'GenericAELG', 'stacks': 30, 'skip': 'd=5,a=0.1'},
    'GAELG30_no_skip':  {'smape': 36.001, 'arch': 'GenericAELG', 'stacks': 30, 'skip': 'none'},
}

# v2 R3 results
v2_r3 = r3.groupby('config_name')['smape'].agg(['mean', 'std']).reset_index()
v2_r3.columns = ['config_name', 'smape', 'std']
v2_top = v2_r3.nsmallest(5, 'smape')

fig, ax = plt.subplots(figsize=(14, 7))

# Plot v1 results
v1_names = list(v1_results.keys())
v1_smapes = [v1_results[k]['smape'] for k in v1_names]
v1_colors = ['#E91E63' if 'GAELG30_no_skip' in k else '#00BCD4' for k in v1_names]

y_offset = 0
for i, (name, data) in enumerate(v1_results.items()):
    color = '#E91E63' if data['smape'] > 20 else '#00BCD4'
    ax.barh(y_offset, data['smape'], color=color, alpha=0.6, edgecolor='white', height=0.7)
    label = f"v1: {name}"
    ax.text(min(data['smape'], 20) + 0.2, y_offset, f"{label} ({data['smape']:.3f})", 
            va='center', fontsize=8)
    y_offset += 1

# Separator
ax.axhline(y_offset - 0.5, color='black', linewidth=1.5, linestyle='-')
ax.text(0.5, y_offset - 0.5, '--- v1 above | v2 below ---', fontsize=9, 
        transform=ax.get_yaxis_transform(), va='center', ha='center', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Plot v2 results
for _, row in v2_top.iterrows():
    arch = r3[r3['config_name'] == row['config_name']]['architecture'].iloc[0]
    ax.barh(y_offset, row['smape'], color=ARCH_COLORS[arch], alpha=0.8, edgecolor='white', height=0.7)
    ax.text(row['smape'] + 0.1, y_offset, 
            f"v2: {row['config_name']} ({row['smape']:.3f} +/- {row['std']:.3f})", 
            va='center', fontsize=8)
    y_offset += 1

ax.set_xlim(0, 20)
ax.set_yticks([])
ax.set_xlabel('SMAPE')
ax.set_title('v1 vs v2 Skip Study: Top Configs Compared')

plt.tight_layout()
plt.show()

print("\nKey comparison:")
print(f"  v1 winner: TW16_skip4_learn     SMAPE=13.521  (TrendAELG+WaveletV3AELG alternating, 16 stacks)")
print(f"  v2 #1:     TWALG30_no_skip       SMAPE={v2_top.iloc[0]['smape']:.3f}  (TrendWaveletAELG unified, 30 stacks, NO skip)")
print(f"  v2 #3:     TWA30_no_skip          SMAPE={v2_top.iloc[2]['smape']:.3f}  (TrendWaveletAE unified, 30 stacks, NO skip)")
print(f"\n  The unified TrendWavelet block matches the v1 alternating winner")
print(f"  with ~3x fewer parameters and without needing skip connections.")
print(f"\n  v1 key finding confirmed: GenericAELG collapses at 30 stacks (SMAPE 36)")
print(f"  v2 new finding: GenericAE does NOT collapse (SMAPE 15.2 at 30 stacks)")

## 11. Convergence and Early Stopping

In [ ]:
# Convergence analysis for R3 configs
r3_conv = r3.groupby('config_name').agg(
    mean_epochs=('epochs_trained', 'mean'),
    std_epochs=('epochs_trained', 'std'),
    mean_smape=('smape', 'mean'),
    mean_time=('training_time_seconds', 'mean'),
    stopping=('stopping_reason', lambda x: x.mode().iloc[0]),
).sort_values('mean_epochs', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Epochs trained
ax = axes[0]
arch_for_cfg = {cfg: r3[r3['config_name']==cfg]['architecture'].iloc[0] for cfg in r3_conv['config_name']}
colors = [ARCH_COLORS[arch_for_cfg[c]] for c in r3_conv['config_name']]

ax.barh(range(len(r3_conv)), r3_conv['mean_epochs'], xerr=r3_conv['std_epochs'],
        capsize=3, color=colors, edgecolor='white')
ax.set_yticks(range(len(r3_conv)))
ax.set_yticklabels(r3_conv['config_name'], fontsize=10)
ax.set_xlabel('Epochs Trained (max 100)')
ax.set_title('R3 Convergence Speed')
ax.invert_yaxis()
for i, ep in enumerate(r3_conv['mean_epochs']):
    ax.text(ep + 1, i, f'{ep:.0f}', va='center', fontsize=9)

# SMAPE vs epochs (scatter)
ax = axes[1]
for _, row in r3_conv.iterrows():
    arch = arch_for_cfg[row['config_name']]
    ax.scatter(row['mean_epochs'], row['mean_smape'], s=120, 
               color=ARCH_COLORS[arch], edgecolors='black', linewidth=0.5, zorder=3)
    ax.annotate(row['config_name'], (row['mean_epochs'], row['mean_smape']),
                fontsize=7, xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Mean Epochs Trained')
ax.set_ylabel('Mean SMAPE')
ax.set_title('Convergence Speed vs Performance')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAll R3 configs early-stopped (none hit max_epochs=100)")
print(f"Fastest converger: {r3_conv.iloc[-1]['config_name']} ({r3_conv.iloc[-1]['mean_epochs']:.0f} epochs)")
print(f"Slowest converger: {r3_conv.iloc[0]['config_name']} ({r3_conv.iloc[0]['mean_epochs']:.0f} epochs)")

## 12. Summary of Findings

### Key Results

| Finding | Evidence |
|---------|----------|
| **TrendWavelet blocks are depth-stable** | SMAPE 13.57 at both 10 and 30 stacks (no degradation) |
| **Skip connections don't help depth-stable architectures** | Skip adds noise to well-behaved residual stream |
| **GenericAE doesn't degrade at depth** | Unlike GenericAELG (v1: SMAPE 36 at 30 stacks), GenericAE stays at 15.2 |
| **Double-VAE is catastrophically bad** | SMAPE 29-44 vs 14-15 for deterministic blocks |
| **Single-VAE + deterministic wavelet works** | SMAPE 15.3 (TVH10), but still worse than TrendWavelet (13.6) |
| **Unified TrendWavelet matches v1 alternating** | SMAPE 13.57 vs 13.52, with 3x fewer parameters |
| **skip_distance=2 is too frequent** | Eliminated in R2 for all architectures |

### Architecture Hierarchy (M4-Yearly)

1. **TrendWaveletAELG / TrendWaveletAE** (SMAPE ~13.6) -- depth-stable, parameter-efficient
2. **GenericAE** (SMAPE ~14.8-15.2) -- depth-stable but lower ceiling
3. **TrendVAE + HaarWaveletV3** (SMAPE ~15.3) -- works but VAE adds cost without benefit
4. **TrendVAE + WaveletV3VAE** (SMAPE ~29-31) -- broken by double-VAE noise
5. **TrendVAE2 + WaveletV3VAE2** (SMAPE ~37-44) -- even worse double-VAE2

### Skip Connection Guidelines (Combined v1 + v2)

- **Use skip** for: GenericAELG at 20+ stacks (rescues from depth collapse)
- **Don't use skip** for: TrendWavelet blocks (depth-stable), GenericAE (depth-stable), any VAE pairing (can't fix fundamental design issue)
- **Optimal skip_distance**: 4-5 for GenericAELG (v1), 3 marginal for TrendWavelet (v2)
- **Optimal skip_alpha**: Fixed 0.1 beats learnable (v1 finding confirmed)